In [0]:
# ============================================================
# Silver — Source 01: RDS PostgreSQL
#
# Transformations:
#   - Cast nanosecond timestamps to proper timestamps
#   - Normalise status enums to lowercase
#   - Deduplicate on primary key (keep latest)
#   - Reject nulls in required fields → quarantine
#   - Validate amounts > 0
#
# Source:  bronze.src_01_orders.*
# Target:  silver.src_01_orders.*
# Quarantine: silver.quarantine.src_01_orders
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable

spark.sql('USE CATALOG bronze')

SILVER_CATALOG = 'silver'
BRONZE_CATALOG = 'bronze'
SOURCE = 'src_01_orders'
QUARANTINE_TABLE = f'{SILVER_CATALOG}.quarantine.src_01_orders'

# Valid status enums per table
VALID_ORDER_STATUSES = ['pending', 'confirmed', 'processing', 'shipped', 'delivered', 'cancelled', 'refunded']
VALID_PAYMENT_STATUSES = ['pending', 'completed', 'failed', 'refunded', 'cancelled']

quarantine_rows = []
quality_report = {}

print('Silver Source 01 — starting...')


In [0]:
# ── ORDERS ────────────────────────────────────────────────────
print('\n--- ORDERS ---')
orders_bronze = spark.table(f'{BRONZE_CATALOG}.src_01_orders.orders')
total = orders_bronze.count()

# Step 1: Cast nanoseconds to timestamp
orders = orders_bronze \
    .withColumn('created_at', (F.col('created_at') / 1e9).cast('timestamp')) \
    .withColumn('updated_at', (F.col('updated_at') / 1e9).cast('timestamp'))

# Step 2: Normalise status to lowercase
orders = orders.withColumn('order_status', F.lower(F.trim(F.col('order_status'))))

# Step 3: Identify bad rows
bad_orders = orders.filter(
    F.col('order_id').isNull() |
    F.col('customer_id').isNull() |
    F.col('total_amount_pence').isNull() |
    (F.col('total_amount_pence') <= 0) |
    ~F.col('order_status').isin(VALID_ORDER_STATUSES)
).withColumn('quarantine_reason', F.lit('failed_validation')) \
 .withColumn('source_table', F.lit('orders'))

bad_count = bad_orders.count()

# Step 4: Good rows — deduplicate on order_id, keep latest
good_orders = orders.filter(
    F.col('order_id').isNotNull() &
    F.col('customer_id').isNotNull() &
    F.col('total_amount_pence').isNotNull() &
    (F.col('total_amount_pence') > 0) &
    F.col('order_status').isin(VALID_ORDER_STATUSES)
)

# Dedup — keep row with latest updated_at per order_id
from pyspark.sql.window import Window
w = Window.partitionBy('order_id').orderBy(F.col('updated_at').desc())
good_orders = good_orders \
    .withColumn('_rn', F.row_number().over(w)) \
    .filter(F.col('_rn') == 1) \
    .drop('_rn')

good_count = good_orders.count()
quality_report['orders'] = {'total': total, 'good': good_count, 'bad': bad_count}
print(f'Orders: {total} total → {good_count} clean, {bad_count} quarantined ({bad_count/total*100:.1f}%)')

# Write good rows to Silver
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {SILVER_CATALOG}.src_01_orders')
if spark.catalog.tableExists(f'{SILVER_CATALOG}.src_01_orders.orders'):
    dt = DeltaTable.forName(spark, f'{SILVER_CATALOG}.src_01_orders.orders')
    dt.alias('t').merge(good_orders.alias('s'), 't.order_id = s.order_id') \
        .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
else:
    good_orders.write.format('delta').mode('overwrite').saveAsTable(f'{SILVER_CATALOG}.src_01_orders.orders')

# Collect bad rows for quarantine
quarantine_rows.append(bad_orders)
print(f'✅ silver.src_01_orders.orders written')


In [0]:
# ── CUSTOMERS ─────────────────────────────────────────────────
print('\n--- CUSTOMERS ---')
customers_bronze = spark.table(f'{BRONZE_CATALOG}.src_01_orders.customers')
total = customers_bronze.count()

customers = customers_bronze \
    .withColumn('created_at', (F.col('created_at') / 1e9).cast('timestamp')) \
    .withColumn('email', F.lower(F.trim(F.col('email')))) \
    .withColumn('email', F.when(F.col('email') == '', None).otherwise(F.col('email')))

bad_customers = customers.filter(
    F.col('customer_id').isNull() |
    F.col('email').isNull()
).withColumn('quarantine_reason', F.lit('failed_validation')) \
 .withColumn('source_table', F.lit('customers'))

good_customers = customers.filter(
    F.col('customer_id').isNotNull() &
    F.col('email').isNotNull()
)

w = Window.partitionBy('customer_id').orderBy(F.col('updated_at').desc())
good_customers = good_customers \
    .withColumn('_rn', F.row_number().over(w)) \
    .filter(F.col('_rn') == 1).drop('_rn')

bad_count = bad_customers.count()
good_count = good_customers.count()
quality_report['customers'] = {'total': total, 'good': good_count, 'bad': bad_count}
print(f'Customers: {total} total → {good_count} clean, {bad_count} quarantined')

if spark.catalog.tableExists(f'{SILVER_CATALOG}.src_01_orders.customers'):
    dt = DeltaTable.forName(spark, f'{SILVER_CATALOG}.src_01_orders.customers')
    dt.alias('t').merge(good_customers.alias('s'), 't.customer_id = s.customer_id') \
        .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
else:
    good_customers.write.format('delta').mode('overwrite').saveAsTable(f'{SILVER_CATALOG}.src_01_orders.customers')

quarantine_rows.append(bad_customers)
print(f'✅ silver.src_01_orders.customers written')


In [0]:
# ── PAYMENTS ──────────────────────────────────────────────────
print('\n--- PAYMENTS ---')
payments_bronze = spark.table(f'{BRONZE_CATALOG}.src_01_orders.payments')
total = payments_bronze.count()

payments = payments_bronze \
    .withColumn('created_at', (F.col('created_at') / 1e9).cast('timestamp')) \
    .withColumn('payment_status', F.lower(F.trim(F.col('payment_status'))))

bad_payments = payments.filter(
    F.col('payment_id').isNull() |
    F.col('order_id').isNull() |
    F.col('amount_pence').isNull() |
    (F.col('amount_pence') <= 0) |
    ~F.col('payment_status').isin(VALID_PAYMENT_STATUSES)
).withColumn('quarantine_reason', F.lit('failed_validation')) \
 .withColumn('source_table', F.lit('payments'))

good_payments = payments.filter(
    F.col('payment_id').isNotNull() &
    F.col('order_id').isNotNull() &
    F.col('amount_pence').isNotNull() &
    (F.col('amount_pence') > 0) &
    F.col('payment_status').isin(VALID_PAYMENT_STATUSES)
)

w = Window.partitionBy('payment_id').orderBy(F.col('updated_at').desc())
good_payments = good_payments \
    .withColumn('_rn', F.row_number().over(w)) \
    .filter(F.col('_rn') == 1).drop('_rn')

bad_count = bad_payments.count()
good_count = good_payments.count()
quality_report['payments'] = {'total': total, 'good': good_count, 'bad': bad_count}
print(f'Payments: {total} total → {good_count} clean, {bad_count} quarantined')

if spark.catalog.tableExists(f'{SILVER_CATALOG}.src_01_orders.payments'):
    dt = DeltaTable.forName(spark, f'{SILVER_CATALOG}.src_01_orders.payments')
    dt.alias('t').merge(good_payments.alias('s'), 't.payment_id = s.payment_id') \
        .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
else:
    good_payments.write.format('delta').mode('overwrite').saveAsTable(f'{SILVER_CATALOG}.src_01_orders.payments')

quarantine_rows.append(bad_payments)
print(f'✅ silver.src_01_orders.payments written')


In [0]:
# ── ORDER ITEMS ───────────────────────────────────────────────
print('\n--- ORDER ITEMS ---')
items_bronze = spark.table(f'{BRONZE_CATALOG}.src_01_orders.order_items')
total = items_bronze.count()

items = items_bronze \
    .withColumn('created_at', (F.col('created_at') / 1e9).cast('timestamp'))

bad_items = items.filter(
    F.col('item_id').isNull() |
    F.col('order_id').isNull() |
    F.col('product_sku').isNull() |
    F.col('quantity').isNull() |
    (F.col('quantity') <= 0) |
    F.col('unit_price_pence').isNull() |
    (F.col('unit_price_pence') <= 0)
).withColumn('quarantine_reason', F.lit('failed_validation')) \
 .withColumn('source_table', F.lit('order_items'))

good_items = items.filter(
    F.col('item_id').isNotNull() &
    F.col('order_id').isNotNull() &
    F.col('product_sku').isNotNull() &
    F.col('quantity').isNotNull() &
    (F.col('quantity') > 0) &
    F.col('unit_price_pence').isNotNull() &
    (F.col('unit_price_pence') > 0)
)

w = Window.partitionBy('item_id').orderBy(F.col('created_at').desc())
good_items = good_items \
    .withColumn('_rn', F.row_number().over(w)) \
    .filter(F.col('_rn') == 1).drop('_rn')

bad_count = bad_items.count()
good_count = good_items.count()
quality_report['order_items'] = {'total': total, 'good': good_count, 'bad': bad_count}
print(f'Order items: {total} total → {good_count} clean, {bad_count} quarantined')

if spark.catalog.tableExists(f'{SILVER_CATALOG}.src_01_orders.order_items'):
    dt = DeltaTable.forName(spark, f'{SILVER_CATALOG}.src_01_orders.order_items')
    dt.alias('t').merge(good_items.alias('s'), 't.item_id = s.item_id') \
        .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
else:
    good_items.write.format('delta').mode('overwrite').saveAsTable(f'{SILVER_CATALOG}.src_01_orders.order_items')

quarantine_rows.append(bad_items)
print(f'✅ silver.src_01_orders.order_items written')


In [0]:
# ── INVENTORY ─────────────────────────────────────────────────
print('\n--- INVENTORY ---')
inv_bronze = spark.table(f'{BRONZE_CATALOG}.src_01_orders.inventory')
total = inv_bronze.count()

inventory = inv_bronze \
    .withColumn('updated_at', (F.col('updated_at') / 1e9).cast('timestamp'))

bad_inv = inventory.filter(
    F.col('product_sku').isNull() |
    F.col('warehouse_id').isNull() |
    F.col('quantity_available').isNull() |
    (F.col('quantity_available') < 0)
).withColumn('quarantine_reason', F.lit('failed_validation')) \
 .withColumn('source_table', F.lit('inventory'))

good_inv = inventory.filter(
    F.col('product_sku').isNotNull() &
    F.col('warehouse_id').isNotNull() &
    F.col('quantity_available').isNotNull() &
    (F.col('quantity_available') >= 0)
)

w = Window.partitionBy('product_sku', 'warehouse_id').orderBy(F.col('updated_at').desc())
good_inv = good_inv \
    .withColumn('_rn', F.row_number().over(w)) \
    .filter(F.col('_rn') == 1).drop('_rn')

bad_count = bad_inv.count()
good_count = good_inv.count()
quality_report['inventory'] = {'total': total, 'good': good_count, 'bad': bad_count}
print(f'Inventory: {total} total → {good_count} clean, {bad_count} quarantined')

if spark.catalog.tableExists(f'{SILVER_CATALOG}.src_01_orders.inventory'):
    dt = DeltaTable.forName(spark, f'{SILVER_CATALOG}.src_01_orders.inventory')
    dt.alias('t').merge(good_inv.alias('s'), 't.product_sku = s.product_sku AND t.warehouse_id = s.warehouse_id') \
        .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
else:
    good_inv.write.format('delta').mode('overwrite').saveAsTable(f'{SILVER_CATALOG}.src_01_orders.inventory')

quarantine_rows.append(bad_inv)
print(f'✅ silver.src_01_orders.inventory written')


In [0]:
# ── QUARANTINE ────────────────────────────────────────────────
print('\n--- QUARANTINE ---')
from functools import reduce

# Union all bad rows — cast all to string first for compatibility
def to_quarantine(df, source_table):
    return df.select(
        F.lit('src_01_orders').alias('source'),
        F.col('source_table'),
        F.col('quarantine_reason'),
        F.lit(None).cast('string').alias('record_id'),
        F.current_timestamp().alias('quarantined_at'),
        F.to_json(F.struct(*[c for c in df.columns if c not in ['quarantine_reason','source_table']])).alias('raw_record')
    )

quarantine_dfs = [to_quarantine(df, 'src_01') for df in quarantine_rows if df.count() > 0]

if quarantine_dfs:
    quarantine_all = reduce(lambda a, b: a.union(b), quarantine_dfs)
    total_quarantined = quarantine_all.count()
    spark.sql(f'CREATE SCHEMA IF NOT EXISTS {SILVER_CATALOG}.quarantine')
    quarantine_all.write.format('delta').mode('append') \
        .option('mergeSchema', 'true') \
        .saveAsTable(QUARANTINE_TABLE)
    print(f'✅ {total_quarantined} rows written to {QUARANTINE_TABLE}')
else:
    print('No quarantine rows — data is clean')


In [0]:
# ── DATA QUALITY REPORT ───────────────────────────────────────
print('\n=== DATA QUALITY REPORT — Source 01 ===')
total_rows = sum(v['total'] for v in quality_report.values())
total_good = sum(v['good'] for v in quality_report.values())
total_bad = sum(v['bad'] for v in quality_report.values())

for table, stats in quality_report.items():
    pct = stats['good']/stats['total']*100 if stats['total'] > 0 else 0
    print(f'  {table}: {stats["good"]}/{stats["total"]} clean ({pct:.1f}%)')

print(f'\n  TOTAL: {total_good}/{total_rows} clean ({total_good/total_rows*100:.1f}%)')
print(f'  QUARANTINED: {total_bad} rows')

print('\n=== SILVER TABLE COUNTS ===')
for t in ['orders', 'customers', 'payments', 'order_items', 'inventory']:
    count = spark.sql(f'SELECT COUNT(*) as cnt FROM {SILVER_CATALOG}.src_01_orders.{t}').collect()[0]['cnt']
    print(f'  silver.src_01_orders.{t}: {count} rows')
